# Instance-level visual search cascade

Experimental notebook: orchestration and result tables only. All logic lives in the `visual_search` package; see `docs/guide.md` for a walkthrough.

**Sections**
1. **Setup:** imports and the main settings.
2. **Datasets:** your labelled test set (from `notebooks/labeling.ipynb`) and a synthetic smoke-test set.
3. **Stage 1:** baselines vs DINOv2 retrieval (metrics per query box).
4. **Per-query inspection:** which queries fail.
5. **Stage 2:** sparse matching + geometric verification variants.
6. **Search the test set:** draw a box on any image and see which test-set boxes are found, with ✓/✗.
7. **Search a new gallery:** coming in the next step (step 6c, part 3).

The interactive sections need JupyterLab with ipympl (`uv run jupyter lab`).

## 1. Setup

This cell switches matplotlib to interactive widgets (for drawing boxes), imports the package and sets the main knobs:

- `TESTSET_DIR`: the folder you labelled with the labeling notebook (default `data/testset/`, or the `VS_TESTSET` environment variable).
- `DINO_MODEL` / `DINO_RESOLUTION`: the stage-1 model. Any `facebook/dinov2-*` checkpoint works; 448 px is the default, and 224 px is about 3× faster.
- `RETRIEVAL_K`: how many candidates stage 1 keeps.
- `SPARSE_K`: how many of those stage 2 re-ranks.
- `CACHE_DIR`: where computed features are stored, so re-runs are fast.

In [ ]:
%matplotlib widget
import logging
import os
from pathlib import Path

import pandas as pd

from visual_search.cascade import Cascade
from visual_search.data import AnnotationSet, load_folder_dataset, make_synthetic_dataset, split_dataset, validate_dataset
from visual_search.data.schema import parse_dataset
from visual_search.device import device_sync
from visual_search.eval import comparison_table, evaluate
from visual_search.masking import BoxMasker, NoMasker
from visual_search.matching import create_matcher
from visual_search.models.dinov2 import Dinov2Backbone
from visual_search.retrieval import POOLINGS, ColorHistogramRetriever, DinoRetriever, RandomRetriever
from visual_search.stages import SparseVerificationStage
from visual_search.ui import SearchApp

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TESTSET_DIR = Path(os.environ.get("VS_TESTSET", ROOT / "data" / "testset"))
FOLDER_DATA_DIR = ROOT / "data"  # older folder-per-object layout, used only if there is no labelled test set
CACHE_DIR = ROOT / "cache"
SEED = 0

DINO_MODEL = "facebook/dinov2-small"
DINO_RESOLUTION = 448
RETRIEVAL_K = 100
SPARSE_K = 20

## 2. Datasets

This cell builds the datasets the tables below are computed on.

- **testset:** your labelled test set, read from `annotations.json`.
  - Every box is one item; images marked *reviewed* without boxes are distractors.
  - For each label, one box (chosen reproducibly from `SEED`) is the **query**. All other boxes and distractors form the **gallery**, and the other boxes of the same label are the ones that must be found.
  - Labels with a single box can't be evaluated and are dropped, with a validator message.
- **folder:** used only if there is no labelled test set: the older folder-per-object layout under `data/`.
- **synthetic:** generated test data, a *smoke test only*. It shows the code works, not which method is better.

Only datasets with at least one query are used further down.

In [ ]:
splits = {}
synthetic_split = split_dataset(
    make_synthetic_dataset(n_instances=12, views_per_instance=3, n_distractors=40, n_near_duplicates=3, seed=SEED),
    seed=SEED,
)

testset_items = []
if (TESTSET_DIR / "annotations.json").exists():
    testset = AnnotationSet.load(TESTSET_DIR)
    print("testset:", testset.summary())
    testset_items = parse_dataset(testset.to_records())
    print(validate_dataset(testset_items).summary())
    testset_split = split_dataset(testset_items, seed=SEED, strict=False)
    if testset_split.queries:
        splits["testset"] = testset_split
elif FOLDER_DATA_DIR.is_dir():
    folder_split = split_dataset(load_folder_dataset(FOLDER_DATA_DIR), seed=SEED, strict=False)
    if folder_split.queries:
        splits["folder"] = folder_split
splits["synthetic"] = synthetic_split
for name, split in splits.items():
    print(f"{name}: {split.summary()}")

## 3. Stage 1: baselines vs DINOv2 pooling modes

This cell loads DINOv2 once and evaluates stage-1-only pipelines on each dataset:

- `random`: random order, the chance level.
- `color_hist`: color histogram, which shows what plain color achieves.
- DINOv2 `cls`: the whole query image, ignoring the box.
- DINOv2 `crop_cls`: the query cropped to its box (the default).
- DINOv2 `masked_mean`: patch features under the box, which makes prompt changes instant.

Gallery items with a box (test-set boxes) are described by their box, and the others by the whole image. Features are cached in `cache/`, so re-runs only pay for the queries.

**Columns:**
- `R@1` / `R@5`: share of query boxes with a correct box at rank 1 / in the top 5.
- `MRR`: mean of 1/rank of the first correct box.
- Latency per query in ms (median / p95).
- `prepare_s`: the offline indexing time.

In [ ]:
backbone = Dinov2Backbone(DINO_MODEL, resolution=DINO_RESOLUTION)
sync = device_sync(backbone.device)
print("device:", backbone.device, "| dim:", backbone.dim, "| patch grid:", backbone.grid)


def stage1_reports(split):
    """Evaluate the baselines and every DINOv2 pooling mode on a split."""
    reports = [evaluate(Cascade(RandomRetriever(seed=SEED)), split), evaluate(Cascade(ColorHistogramRetriever()), split)]
    for pooling in POOLINGS:
        retriever = DinoRetriever(backbone, pooling=pooling, cache_dir=CACHE_DIR)
        reports.append(evaluate(Cascade(retriever, retrieval_k=RETRIEVAL_K, sync=sync), split))
    return reports


reports = {name: stage1_reports(split) for name, split in splits.items()}
tables = {}
for name, reps in reports.items():
    t = comparison_table(reps).drop(columns=["stage"])
    t["prepare_s"] = [sum(r.prepare_s.values()) for r in reps]
    tables[name] = t
pd.concat(tables, names=["dataset"]).round(3)

## 4. Per-query inspection

One row per query box for DINOv2 `crop_cls` on your test set (or the synthetic set if there is none):
- `rank@retrieval`: the rank at which the first correct box was found (empty = not found);
- `top1@retrieval`: what came first instead.

Sorted worst first, so the failures to look at are on top. Copy a `query_id` into the search section below to inspect one interactively.

In [ ]:
inspect_name = "testset" if "testset" in reports else next(iter(reports))
best = next(r for r in reports[inspect_name] if "/crop_cls" in r.name)
print(f"{inspect_name}: {best.name}")
best.per_query.sort_values("rank@retrieval", ascending=False, na_position="first")

## 5. Stage 2: sparse matching + geometric verification

DINOv2 `crop_cls` retrieves the top `RETRIEVAL_K`. Then the top `SPARSE_K` are re-ranked by the number of geometrically verified keypoint matches. The query side uses the query box; the gallery side uses the gallery box (test set) or the whole image.

Variants compared:
- **estimator:** OpenCV USAC MAGSAC++ vs classic RANSAC;
- **model:** homography `H` (flat objects), affine `A`, fundamental `F` (3D, no warp);
- **ablations:** no masks, plain SIFT instead of RootSIFT;
- **learned matchers:** ALIKED + LightGlue and XFeat + LighterGlue. They need `uv sync --extra sparse` and re-rank only the top 10, since they cost about 35–90 ms per pair on an M2.

The synthetic set is flat and easy (it saturates at R@1 = 1), so only your test set shows real differences.

In [ ]:
LEARNED = {}
for _name in ["aliked-lightglue", "xfeat-lighterglue"]:
    try:
        LEARNED[_name] = create_matcher(_name)
    except ImportError as exc:
        print(f"skipping {_name}: {exc}")


def stage2_configs():
    """(name, stages) pairs for the stage-2 ablation."""
    sparse = lambda **kw: SparseVerificationStage(top_k=SPARSE_K, cache_dir=CACHE_DIR, **kw)
    return [
        ("stage 1 only", []),
        ("sift H magsac", [sparse(matcher="sift", model_type="H", method="magsac")]),
        ("sift H ransac", [sparse(matcher="sift", model_type="H", method="ransac")]),
        ("sift A magsac", [sparse(matcher="sift", model_type="A")]),
        ("sift F magsac", [sparse(matcher="sift", model_type="F")]),
        ("sift H magsac, no mask", [sparse(matcher="sift", use_masks=False, name="sift-nomask")]),
        ("plain SIFT H magsac", [sparse(matcher=create_matcher("sift", root_sift=False), name="sift-plain")]),
    ] + [
        (f"{name} H magsac (top-10)", [SparseVerificationStage(LEARNED[name], top_k=10, cache_dir=CACHE_DIR)])
        for name in LEARNED
    ]


stage2_tables = {}
for name, split in splits.items():
    reps = []
    for label, stages in stage2_configs():
        retriever = DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR)
        reps.append(evaluate(Cascade(retriever, RETRIEVAL_K, stages, sync=sync), split, name=label))
    stage2_tables[name] = comparison_table(reps)
pd.concat(stage2_tables, names=["dataset"]).round(3)

## 6. Search the test set

This cell builds an interactive search app whose **gallery is every box in your test set**, plus the reviewed images without boxes as distractors. It uses the default pipeline: DINOv2 `crop_cls`, then SIFT verification of the top `SPARSE_K`. Without a labelled test set it falls back to the synthetic set.

The first run indexes all test-set boxes. That takes a while once, and then it's cached.

In [ ]:
if testset_items:
    search_gallery, search_queries = testset_items, [it for it in testset_items if it.label]
else:
    print("No labelled test set found; using the synthetic set.")
    search_gallery, search_queries = synthetic_split.gallery, synthetic_split.queries

MASKER = BoxMasker()  # NoMasker() = ablation without masks
search_cascade = Cascade(
    DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR),
    retrieval_k=RETRIEVAL_K,
    stages=[SparseVerificationStage("sift", top_k=SPARSE_K, model_type="H", method="magsac", cache_dir=CACHE_DIR)],
    sync=sync,
)
testset_app = SearchApp(search_cascade, gallery=search_gallery, queries=search_queries, masker=MASKER, top_k=5)
testset_app.widget

**How to use it**

1. **Pick a query.** Either choose a labelled box from the **Query** dropdown (`image#n [label]`), which loads its image with that box, or type the path of any image and press **Load path**.
2. **Draw or adjust the box.** In *Box* mode drag a rectangle; *Points* mode takes clicks (left = on the object, right or shift+click = exclude). The blue overlay previews the mask.
3. **Check the Expected label.**
   - For a test-set box it is filled in automatically.
   - For a new image, type or pick the label of the object you drew (it autocompletes from the test set) to get ✓/✗ marks. Leave it empty to just see the ranking.
4. **Press Search.** You get one row per stage with the top 5:
   - each result is shown as a crop around its box (yellow), with its score, its label and a green ✓ / red ✗ frame;
   - the query's own box is never returned when you picked it from the test set.
5. **Pick a result in Detail** to see the keypoint matches (green = inliers) and the gallery box warped onto your query.

`testset_app.last_result` holds the full rankings and score components after a search.

The masker is a box mask for now. Once SAM3 fp16 passes `scripts/try_sam3.py`, switch to:
```python
from visual_search.masking.sam3 import Sam3Config, Sam3Masker
MASKER = Sam3Masker(Sam3Config(model_dir=ROOT / "models/sam3", variant="fp16"))
```

## 7. Search a new gallery

*Coming in step 6c, part 3.* You'll point at a folder, a glob or a list of files, draw a box on a query and get the top-n regions where the object appears. The object is found in each gallery image with DINOv2, so the gallery needs no labels. There are no ✓/✗ marks; you judge the results yourself.